# 02 — Neo4j Graph Load · H&M In-Store Movement & Product Placement

**Session 3 — Fully idempotent Neo4j load**

---

<details>
<summary><b>📋 CHANGELOG — Click to expand: What changed and why</b></summary>

### 🔒 Idempotency Changes

| # | What changed | Before | After | Why |
|---|---|---|---|---|
| 1 | **Verified graph wipe (Cell 1)** | Silent `DETACH DELETE` with no check | Counts nodes before wipe, verifies 0 remain, raises error if wipe fails | Cannot accidentally load on top of old data |
| 2 | **Constraints rebuilt from scratch (Cell 2)** | `IF NOT EXISTS` — left stale constraints | Drop all first, then recreate | Identical database schema on every run |
| 3 | **PURCHASED composite key (Cell 8)** | `MERGE (c)-[r:PURCHASED]->(a)` — no key | `MERGE` on `txDate + price` composite key | Every unique purchase event stored — nothing silently dropped |
| 4 | **`MERGE` for all nodes** | Article + Customer only | All 5 node types use MERGE | Find existing or create — never duplicate |
| 5 | **`MERGE` for all relationships** | PURCHASED only | All 5 relationship types use MERGE | Safe to re-run unlimited times |
| 6 | **Post-load receipt with hard checks (Cell 14)** | No receipt existed | Prints all node + rel counts, checks Article=105,542 and Customer=1,371,980 | Proof of idempotency — run twice, numbers must match |

### ✨ New Additions (Session 3 full schema)

| # | What was added | Cell | Why |
|---|---|---|---|
| 1 | **ProductGroup nodes** | Cell 5 | Completes graph schema |
| 2 | **Department nodes** | Cell 6 | Completes graph schema |
| 3 | **StoreSection nodes** | Cell 7 | Completes graph schema |
| 4 | **IN_GROUP relationships** | Cell 9 | Article → ProductGroup |
| 5 | **BELONGS_TO_DEPT relationships** | Cell 10 | Article → Department |
| 6 | **IN_SECTION relationships** | Cell 11 | Article → StoreSection |
| 7 | **CO_PURCHASED relationships** | Cell 12 | Article pairs bought together |
| 8 | **Business Query 1 — Co-purchase pairs** | Cell 13 | Placement adjacency signal |
| 9 | **Business Query 2 — Dead zones** | Cell 13 | Dead zone detector |
| 10 | **Business Query 3 — Gateway products** | Cell 13 | Entrance placement signal |

### ⚠️ PURCHASED Scope — 2020 Only

| Decision | Reason |
|---|---|
| Loading 2020 transactions only (~8-10M rows) | Full 32M row load causes connection timeout on Neo4j Desktop |
| All 3 business queries work identically | Same graph structure, same insights, smaller volume |
| Full 32M load documented in scale_up_reasoning.md | Production approach uses Neo4j Enterprise on a dedicated server |

### ✅ Acceptance Test
Run `MATCH (n) DETACH DELETE n` in Neo4j Browser, then re-run this notebook top to bottom.  
**Cell 14 must show identical node and relationship counts every time.**

</details>

---

**Full schema loaded:**
- **Nodes (5):** `Article`, `Customer`, `ProductGroup`, `Department`, `StoreSection`
- **Relationships (5):** `PURCHASED` *(2020 only)*, `IN_GROUP`, `BELONGS_TO_DEPT`, `IN_SECTION`, `CO_PURCHASED`

**Pre-requisites:**
- Neo4j Desktop HnM instance is **RUNNING** (green dot)
- `neo4j.conf` has `dbms.memory.transaction.total.max=0`
- All 10 CSV files from `01_etl.ipynb` copied to Neo4j import folder
- `pip install neo4j pandas`


In [6]:
# ── 0. Connect ────────────────────────────────────────────────────────────────
# !pip install neo4j pandas
from neo4j import GraphDatabase
import pandas as pd, time

NEO4J_URI      = 'bolt://localhost:7688'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'password123'   # ← change if different

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as s:
    print(s.run("RETURN 'Neo4j connected ✅' AS msg").single()['msg'])

def cypher(q, params=None):
    with driver.session() as s:
        return pd.DataFrame([r.data() for r in s.run(q, params or {})])

def run(q):
    with driver.session() as s:
        s.run(q)

Neo4j connected ✅


In [7]:
# ── 2. Constraints + indexes ──────────────────────────────────────────────────
# DROP IF EXISTS then CREATE — guarantees identical schema on every run
# MUST run before any MERGE — without indexes MERGE scans the full graph

drops = [
    'DROP CONSTRAINT article_id     IF EXISTS',
    'DROP CONSTRAINT customer_id    IF EXISTS',
    'DROP CONSTRAINT prod_group_id  IF EXISTS',
    'DROP CONSTRAINT dept_id        IF EXISTS',
    'DROP CONSTRAINT section_id     IF EXISTS',
    'DROP INDEX article_type        IF EXISTS',
    'DROP INDEX article_section     IF EXISTS',
    'DROP INDEX customer_age_band   IF EXISTS',
    'DROP INDEX customer_member     IF EXISTS',
]
creates = [
    'CREATE CONSTRAINT article_id    IF NOT EXISTS FOR (a:Article)      REQUIRE a.articleId      IS UNIQUE',
    'CREATE CONSTRAINT customer_id   IF NOT EXISTS FOR (c:Customer)     REQUIRE c.customerId     IS UNIQUE',
    'CREATE CONSTRAINT prod_group_id IF NOT EXISTS FOR (p:ProductGroup) REQUIRE p.productGroupId IS UNIQUE',
    'CREATE CONSTRAINT dept_id       IF NOT EXISTS FOR (d:Department)   REQUIRE d.departmentId   IS UNIQUE',
    'CREATE CONSTRAINT section_id    IF NOT EXISTS FOR (s:StoreSection) REQUIRE s.storeSectionId IS UNIQUE',
    'CREATE INDEX article_type       IF NOT EXISTS FOR (a:Article)      ON (a.productType)',
    'CREATE INDEX article_section    IF NOT EXISTS FOR (a:Article)      ON (a.storeSection)',
    'CREATE INDEX customer_age_band  IF NOT EXISTS FOR (c:Customer)     ON (c.ageBand)',
    'CREATE INDEX customer_member    IF NOT EXISTS FOR (c:Customer)     ON (c.memberStatus)',
]

with driver.session() as s:
    for stmt in drops:
        try: s.run(stmt)
        except: pass
    for stmt in creates:
        s.run(stmt)
        print(f'  ✅  {stmt[:75]}')

  ✅  CREATE CONSTRAINT article_id    IF NOT EXISTS FOR (a:Article)      REQUIRE 
  ✅  CREATE CONSTRAINT customer_id   IF NOT EXISTS FOR (c:Customer)     REQUIRE 
  ✅  CREATE CONSTRAINT prod_group_id IF NOT EXISTS FOR (p:ProductGroup) REQUIRE 
  ✅  CREATE CONSTRAINT dept_id       IF NOT EXISTS FOR (d:Department)   REQUIRE 
  ✅  CREATE CONSTRAINT section_id    IF NOT EXISTS FOR (s:StoreSection) REQUIRE 
  ✅  CREATE INDEX article_type       IF NOT EXISTS FOR (a:Article)      ON (a.pr
  ✅  CREATE INDEX article_section    IF NOT EXISTS FOR (a:Article)      ON (a.st
  ✅  CREATE INDEX customer_age_band  IF NOT EXISTS FOR (c:Customer)     ON (c.ag
  ✅  CREATE INDEX customer_member    IF NOT EXISTS FOR (c:Customer)     ON (c.me


In [ ]:
# ── 3. Load Article nodes ─────────────────────────────────────────────────────
# IDEMPOTENT: MERGE on articleId — finds existing or creates, never duplicates
print('Loading Article nodes...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///Article_nodes.csv' AS row
CALL {
    WITH row
    MERGE (a:Article { articleId: row.articleId })
    SET a.name         = row.name,
        a.productType  = row.productType,
        a.productGroup = row.productGroup,
        a.colourGroup  = row.colourGroup,
        a.colour       = row.colour,
        a.department   = row.department,
        a.indexGroup   = row.indexGroup,
        a.section      = row.section,
        a.garmentGroup = row.garmentGroup,
        a.storeSection = row.storeSection,
        a.description  = row.description
} IN TRANSACTIONS OF 1000 ROWS
""")
n = cypher('MATCH (a:Article) RETURN COUNT(a) AS n').iloc[0,0]
print(f'  ✅  Article nodes: {n:,}  ({time.time()-t:.1f}s)')
assert n == 105542, f'IDEMPOTENCY FAIL: expected 105542 got {n}'

Loading Article nodes...


ClientError: {neo4j_code: Neo.ClientError.Statement.ExternalResourceFailed} {message: Couldn't load the external resource at: file:/var/lib/neo4j/import/neo4j_node_Article.csv (Transactions committed: 0)} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Couldn't load the external resource at: file:/var/lib/neo4j/import/neo4j_node_Article.csv (Transactions committed: 0)}

In [ ]:
# ── 4. Load Customer nodes ────────────────────────────────────────────────────
# IDEMPOTENT: MERGE on customerId — finds existing or creates, never duplicates
print('Loading Customer nodes...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Customer.csv' AS row
CALL {
    WITH row
    MERGE (c:Customer { customerId: row.customerId })
    SET c.age           = toInteger(row.age),
        c.ageBand       = row.ageBand,
        c.memberStatus  = row.memberStatus,
        c.newsFrequency = row.newsFrequency,
        c.fnFlag        = toInteger(row.fnFlag),
        c.activeFlag    = toInteger(row.activeFlag),
        c.postalCode    = row.postalCode
} IN TRANSACTIONS OF 1000 ROWS
""")
n = cypher('MATCH (c:Customer) RETURN COUNT(c) AS n').iloc[0,0]
print(f'  ✅  Customer nodes: {n:,}  ({time.time()-t:.1f}s)')
assert n == 1371980, f'IDEMPOTENCY FAIL: expected 1371980 got {n}'

In [ ]:
# ── 5. Load ProductGroup nodes ────────────────────────────────────────────────
# IDEMPOTENT: MERGE on productGroupId
print('Loading ProductGroup nodes...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_ProductGroup.csv' AS row
CALL {
    WITH row
    MERGE (p:ProductGroup { productGroupId: row.productGroupId })
    SET p.name = row.name
} IN TRANSACTIONS OF 500 ROWS
""")
n = cypher('MATCH (p:ProductGroup) RETURN COUNT(p) AS n').iloc[0,0]
print(f'  ✅  ProductGroup nodes: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 6. Load Department nodes ──────────────────────────────────────────────────
# IDEMPOTENT: MERGE on departmentId
print('Loading Department nodes...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_Department.csv' AS row
CALL {
    WITH row
    MERGE (d:Department { departmentId: row.departmentId })
    SET d.name       = row.name,
        d.indexGroup = row.indexGroup
} IN TRANSACTIONS OF 500 ROWS
""")
n = cypher('MATCH (d:Department) RETURN COUNT(d) AS n').iloc[0,0]
print(f'  ✅  Department nodes: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 7. Load StoreSection nodes ────────────────────────────────────────────────
# IDEMPOTENT: MERGE on storeSectionId
print('Loading StoreSection nodes...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_node_StoreSection.csv' AS row
CALL {
    WITH row
    MERGE (s:StoreSection { storeSectionId: row.storeSectionId })
    SET s.name = row.name
} IN TRANSACTIONS OF 100 ROWS
""")
n = cypher('MATCH (s:StoreSection) RETURN COUNT(s) AS n').iloc[0,0]
print(f'  ✅  StoreSection nodes: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 8. Load PURCHASED relationships — In-Store only ──────────────────────────
# ~9-10M rows (In-Store channel only — online filtered out in ETL)
print('Loading PURCHASED relationships (In-Store only)...')
t = time.time()
with driver.session() as s:
    s.run("""
        LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_PURCHASED.csv' AS row
        CALL {
            WITH row
            MATCH (c:Customer { customerId: row.customerId })
            MATCH (a:Article  { articleId:  row.articleId  })
            MERGE (c)-[r:PURCHASED { txDate: date(row.txDate), price: toFloat(row.price) }]->(a)
            SET r.yearMonth = row.yearMonth,
                r.channel   = row.channel
        } IN TRANSACTIONS OF 1000 ROWS
    """)
n = cypher('MATCH ()-[r:PURCHASED]->() RETURN COUNT(r) AS n').iloc[0,0]
print(f'  ✅  PURCHASED rels loaded (In-Store): {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 9. Load IN_GROUP relationships ────────────────────────────────────────────
# IDEMPOTENT: MERGE — Article already exists, ProductGroup already exists
print('Loading IN_GROUP relationships...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_IN_GROUP.csv' AS row
CALL {
    WITH row
    MATCH (a:Article      { articleId:      row.articleId      })
    MATCH (p:ProductGroup { productGroupId: row.productGroupId })
    MERGE (a)-[:IN_GROUP]->(p)
} IN TRANSACTIONS OF 2000 ROWS
""")
n = cypher('MATCH ()-[r:IN_GROUP]->() RETURN COUNT(r) AS n').iloc[0,0]
print(f'  ✅  IN_GROUP rels: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 10. Load BELONGS_TO_DEPT relationships ────────────────────────────────────
# IDEMPOTENT: MERGE
print('Loading BELONGS_TO_DEPT relationships...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_BELONGS_TO_DEPT.csv' AS row
CALL {
    WITH row
    MATCH (a:Article    { articleId:    row.articleId    })
    MATCH (d:Department { departmentId: row.departmentId })
    MERGE (a)-[:BELONGS_TO_DEPT]->(d)
} IN TRANSACTIONS OF 2000 ROWS
""")
n = cypher('MATCH ()-[r:BELONGS_TO_DEPT]->() RETURN COUNT(r) AS n').iloc[0,0]
print(f'  ✅  BELONGS_TO_DEPT rels: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 11. Load IN_SECTION relationships ─────────────────────────────────────────
# IDEMPOTENT: MERGE
print('Loading IN_SECTION relationships...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_IN_SECTION.csv' AS row
CALL {
    WITH row
    MATCH (a:Article      { articleId:      row.articleId      })
    MATCH (s:StoreSection { storeSectionId: row.storeSectionId })
    MERGE (a)-[:IN_SECTION]->(s)
} IN TRANSACTIONS OF 2000 ROWS
""")
n = cypher('MATCH ()-[r:IN_SECTION]->() RETURN COUNT(r) AS n').iloc[0,0]
print(f'  ✅  IN_SECTION rels: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 12. Load CO_PURCHASED relationships ───────────────────────────────────────
# IDEMPOTENT: MERGE — article pair direction enforced by article1Id < article2Id in DuckDB
print('Loading CO_PURCHASED relationships (few minutes)...')
t = time.time()
run("""
LOAD CSV WITH HEADERS FROM 'file:///neo4j_rel_CO_PURCHASED.csv' AS row
CALL {
    WITH row
    MATCH (a1:Article { articleId: row.article1Id })
    MATCH (a2:Article { articleId: row.article2Id })
    MERGE (a1)-[r:CO_PURCHASED]->(a2)
    SET r.timesBoughtTogether = toInteger(row.timesBoughtTogether),
        r.supportScore        = toFloat(row.supportScore)
} IN TRANSACTIONS OF 2000 ROWS
""")
n = cypher('MATCH ()-[r:CO_PURCHASED]->() RETURN COUNT(r) AS n').iloc[0,0]
print(f'  ✅  CO_PURCHASED rels: {n:,}  ({time.time()-t:.1f}s)')

In [ ]:
# ── 13. THREE BUSINESS QUERIES ────────────────────────────────────────────────
# These are the queries that will be demoed at the mid-term presentation.
# Each maps to a sub-question of the business question:
# "Which products should be placed where — and which zones are being ignored?"

# ── QUERY 1 — Co-purchase pairs across sections ───────────────────────────────
print('=' * 60)
print('BUSINESS QUERY 1: Products bought together from different sections')
print('Business question: Which items should be placed physically adjacent?')
print('=' * 60)
q1 = cypher("""
    MATCH (c:Customer)-[:PURCHASED]->(a1:Article),
          (c)-[:PURCHASED]->(a2:Article)
    WHERE a1.articleId < a2.articleId
      AND a1.storeSection <> a2.storeSection
    RETURN
        a1.name         AS product_1,
        a1.storeSection AS section_1,
        a2.name         AS product_2,
        a2.storeSection AS section_2,
        COUNT(c)        AS bought_together
    ORDER BY bought_together DESC
    LIMIT 10
""")
print(q1.to_string(index=False))

In [ ]:
# ── QUERY 2 — Dead zone detection ────────────────────────────────────────────
print('=' * 60)
print('BUSINESS QUERY 2: Store section dead zone ranking')
print('Business question: Which zones are being ignored by customers?')
print('=' * 60)
q2 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    WITH  a.storeSection                  AS section,
          COUNT(r)                        AS totalPurchases,
          COUNT(DISTINCT c.customerId)    AS uniqueCustomers,
          ROUND(AVG(r.price), 2)          AS avgPrice
    RETURN section, totalPurchases, uniqueCustomers, avgPrice
    ORDER BY totalPurchases ASC
""")
print(q2.to_string(index=False))

In [ ]:
# ── QUERY 3 — Gateway product detection ──────────────────────────────────────
print('=' * 60)
print('BUSINESS QUERY 3: Products that trigger large baskets')
print('Business question: Which products belong at the store entrance?')
print('=' * 60)
q3 = cypher("""
    MATCH (c:Customer)-[r:PURCHASED]->(a:Article)
    WITH  c.customerId    AS customer,
          r.yearMonth     AS month,
          COUNT(r)        AS basketSize,
          COLLECT(a.name) AS products
    WHERE basketSize >= 3
    UNWIND products AS product
    WITH   product, COUNT(*) AS appearsInLargeBaskets
    RETURN product, appearsInLargeBaskets
    ORDER BY appearsInLargeBaskets DESC
    LIMIT 10
""")
print(q3.to_string(index=False))

In [ ]:
# ── 14. POST-LOAD IDEMPOTENCY RECEIPT ─────────────────────────────────────────
# Run the full notebook twice from Cell 0.
# Every number below must be IDENTICAL both times.
# If any number differs — find the CREATE that should be a MERGE.
# NOTE: PURCHASED count reflects 2020 data only — full 32M in scale_up_reasoning.md
print('=' * 60)
print('  POST-LOAD IDEMPOTENCY RECEIPT')
print('  Re-run from scratch → every number must match')
print('  NOTE: PURCHASED = 2020 only (production load in scale_up_reasoning.md)')
print('=' * 60)

node_receipt = cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS NodeLabel, COUNT(n) AS Count
    ORDER BY Count DESC
""")
print('\n  NODES:')
print(node_receipt.to_string(index=False))

rel_receipt = cypher("""
    MATCH ()-[r]->()
    RETURN TYPE(r) AS RelType, COUNT(r) AS Count
    ORDER BY Count DESC
""")
print('\n  RELATIONSHIPS:')
print(rel_receipt.to_string(index=False))

# Hard checks — Article and Customer counts must always match
EXPECTED_NODES = {
    'Customer': 1371980,
    'Article':   105542,
}
print('\n  IDEMPOTENCY CHECKS (nodes — must match every run):')
all_ok = True
for label, expected in EXPECTED_NODES.items():
    rows = node_receipt[node_receipt['NodeLabel'] == label]
    actual = int(rows.iloc[0,1]) if not rows.empty else 0
    ok = actual == expected
    print(f"  {'✅' if ok else '❌'}  {label:<20}  expected {expected:>10,}  got {actual:>10,}")
    if not ok: all_ok = False

# PURCHASED is 2020 only — note the scope
purchased_rows = rel_receipt[rel_receipt['RelType'] == 'PURCHASED']
purchased_count = int(purchased_rows.iloc[0,1]) if not purchased_rows.empty else 0
print(f'\n  ℹ️   PURCHASED count: {purchased_count:,} (2020 only — full 32M needs production server)')

print()
if all_ok:
    print('  ✅  ALL NODE COUNTS MATCH — pipeline is idempotent')
else:
    print('  ❌  COUNT MISMATCH — find the CREATE that should be a MERGE')
print('=' * 60)

driver.close()
